In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append('../..')
from datetime import datetime

import os
import yaml
from pyspark.sql.functions import to_date, date_sub
from urllib.parse import urlparse
import lib_trip_spend.spark_general_utilities as util_func

from databricks.feature_engineering import FeatureEngineeringClient


In [0]:
today_str = datetime.today().strftime("%Y-%m-%d")
dbutils.widgets.text("run_as_date", today_str, "Date for data processing") # create the widget if missing

In [0]:
date_of_run_str = dbutils.widgets.get("run_as_date")
run_as_date = datetime.strptime(date_of_run_str, "%Y-%m-%d").date() 

print(f"Run as date:    {run_as_date}")

In [0]:
config_path = "config/config.yml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Missing configuration: {config_path}") 

with open(config_path, "r") as config_file:
    config = yaml.load(config_file, Loader=yaml.FullLoader)


RUN_NAME                            = config["shared"]["run_name"]          # "prod_2025_09_28"

LAST_FISCAL_WEEK_TRAINING           = config["etl"]["end_date"]
FIRST_FISCAL_WEEK_TRAINING          = config["etl"]["start_date"]

In [0]:
import mlflow
mlflow.autolog(disable=True)

In [0]:
customers = spark.table(model_trip_spend_etl_intermediate)  

# If you need a archive version you can pull like this: 
#------------------------------------------------------------------------------------------
# customers = spark.table(model_trip_spend_etl_intermediate_archive).filter(
#         (f.col("run_date") == f.lit(run_as_date).cast("date")) &
#         (f.col("run_name") == RUN_NAME) &
#         (f.col("last_fiscal_week_training") == LAST_FISCAL_WEEK_TRAINING) &
#         (f.col("first_fiscal_week_training") == FIRST_FISCAL_WEEK_TRAINING)
#     )

In [0]:
print("----------------2/4 read in features----------------")

features = spark.read.csv(path=feature_path, header=True) # <------------- this is where we are pulling the csv from volume 

categorical_features, continious_features = util_func.get_features_list(
    features
)

In [0]:
# Transform data
print("----------------3/4 transforming data----------------")
# print(f"----categorical_features = {categorical_features} \n \n")
# print(f"----continious_features = {continious_features} \n \n")
# print(f"----customers columns = {customers.columns} \n \n ")

transformed_customer_data = util_func.prepare_data( 
    customers, categorical_features, continious_features
)

In [0]:
# 1. Write the transformed data to the file first.
# The expensive 'prepare_data' transformation is executed only here.

#spark.sql(f"DELETE FROM {model_trip_spend_etl_output}")

transformed_customer_data.write.mode("overwrite").saveAsTable(model_trip_spend_etl_output)

# # 2. Read the data back to get the count for logging. This is a very fast operation.
output = spark.table(model_trip_spend_etl_output)
row_count = output.count()
print(
    f"----------------Successfully wrote {row_count} rows----------------"
)

In [0]:
df_w_additional_columns = (
    output
    .withColumn("run_date", f.lit(run_as_date).cast('date'))
    .withColumn("run_name", f.lit(RUN_NAME).cast("string"))
    .withColumn("last_fiscal_week_training", f.lit(LAST_FISCAL_WEEK_TRAINING).cast("string"))
    .withColumn("first_fiscal_week_training", f.lit(FIRST_FISCAL_WEEK_TRAINING).cast("string"))
) 


df_w_additional_columns.write.mode("overwrite").option(
    "replaceWhere",
    f"run_date = '{run_as_date}' AND run_name = '{RUN_NAME}' AND last_fiscal_week_training = '{LAST_FISCAL_WEEK_TRAINING}' AND first_fiscal_week_training = '{FIRST_FISCAL_WEEK_TRAINING}'"
).saveAsTable(model_trip_spend_etl_output_archive)